# Lid-Driven Cavity (2D)

A square box of fluid with no-slip walls whose **top wall is dragged sideways**
at unit speed. The shear from the lid rolls the whole cavity into one primary
vortex, with weaker counter-rotating corner vortices underneath it; the flow
reaches a steady state, which is why this is the classic incompressible
benchmark and why `tLimit` here is long (30 s) rather than a few oscillations.

The lid is **not** a moving boundary region. It is a dynamic Dirichlet
condition on the velocity: `initialConditions` installs a
`BoundaryCondition(type=dynamic, dirichletFunctions={'velocities': ...})` that
overwrites `u` with `lidVelocity` for every particle above `lidHeight`, and
`enforceDirichlet` applies it once before the run so the initial state is
consistent with it. That is the piece to read (and to copy) if you want a
different driving condition -- it is eight lines in
`warpSPH/cases/lidDrivenCavity.py` and nothing else in the case knows about the
lid.

Walls come from `boundaryRegion(domainBoundarySdf(ctx), kind=BCType.noSlip)`:
the domain is `band = 5` particle layers wider than the interior, and the
walls are what fills that band. `L` is the *interior* side.

![](outputs/09-lidDrivenCavity.gif)


## Every knob, and what it does

The parameters cell below is the whole command line of `09-lid-driven-cavity.py` written out:
`CaseSpec` fields first, then `lidDrivenCavityCase.params` -- the case's own physics knobs,
each of which is also a `--flag`. Anything not named there keeps the value in
`lidDrivenCavityCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `128` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D |
| `L` | `2.0` | side of the (periodic) box |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `30.0` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `lidVelocity` | `1.0` | speed the top wall is dragged at, +x |
| `lidHeight` | `1.0` | every particle above this y is held at `lidVelocity`; the interior top is at `L/2 = 1.0` |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.0005` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit |
| `inviscid`, `nu` | `True`, `0.0` | physical viscosity: `inviscid=True` leaves the scheme's own dissipation as the only one |
| `freeSurface` | `False` | surface detection, on for a case with a free surface |
| `band` | `5` | particle layers of boundary padding around the domain |
| `markerSize` | `8` | plot only: particle marker size |


**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `lidDrivenCavityCase.initialConditions(ctx, system)`.
   That is where `setupWeaklyCompressibleTimestep` picks the sound speed and
   `config.dt` *together* from `targetDt` -- weakly compressible SPH is free to
   choose its own stiffness, so the timestep is fixed first and `c0` follows
   from the acoustic CFL. Skip it and `config.dt` stays `None`, and the lid condition is never installed -- `initialConditions` is also where the Dirichlet boundary condition is built and enforced.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `lidDrivenCavityCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-Hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.lidDrivenCavity import lidDrivenCavityCase
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `09-lid-driven-cavity.py`, made explicit and editable here -- the table in the
# intro cell says what each one does. `lidDrivenCavityCase.defaults`/`.params` are
# the same values the CLI script starts from.
spec = CaseSpec(caseName=lidDrivenCavityCase.name, scheme=lidDrivenCavityCase.scheme,
                params=dict(lidDrivenCavityCase.params)) \
    .merged(**lidDrivenCavityCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=128,
    dim=2,
    # Interior side; the simulated box is `band` particle layers wider on
    # every side, and the walls are sampled from that band.
    L=2.0,

    # --- time stepping ---------------------------------------------------
    # Long: the primary vortex has to reach steady state.
    tLimit=30.0,

    # --- output --------------------------------------------------------------
    caseName='09-lidDrivenCavity',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the cavity's own knobs -----------------------------------------------
    params=dict(
        # the lid
        lidVelocity=1.0, lidHeight=1.0,
        # the walls: 5 particle layers of no-slip boundary around the interior
        band=5,
        # the fluid
        rho0=1.0, targetDt=0.0005, inviscid=True, nu=0.0,
        markerSize=8,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`lidDrivenCavityCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have:
# the lid Dirichlet condition is installed and enforced there, and
# it is where the sound speed and `config.dt` are chosen together from
# `targetDt`, so skipping it leaves `config.dt` unset.
ctx = buildContext(lidDrivenCavityCase, spec)
lidDrivenCavityCase.configureScheme(ctx)
system = lidDrivenCavityCase.buildSystem(ctx)
lidDrivenCavityCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')

In [ ]:
# What was actually built: the sampled regions, fluid and boundary, against the
# domain (black) the run is periodic in. This is the cell to look at when a
# geometry parameter above did something other than what it sounded like.
figure, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["regions"])} regions, '
                     f'{len(runningState.state.positions)} particles')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not lidDrivenCavityCase.setupPlot -- see the intro cell
# for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = lidDrivenCavityCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=lidDrivenCavityCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []

for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = lidDrivenCavityCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=lidDrivenCavityCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Centreline profiles

The two profiles below are what a lid-driven cavity is usually compared on --
Ghia, Ghia & Shin's tabulated `u(y)` and `v(x)` at a given Reynolds number are
the reference, and the shape (one steep shear layer under the lid, a broad
return flow beneath it, a sign change near each wall) is readable without them.
Note this run is *inviscid* apart from the scheme's own dissipation, so there
is no Reynolds number to match: set `inviscid=False, nu=...` in the parameters
cell to get one.

In [ ]:
# The standard lid-driven-cavity diagnostic: the velocity profiles through the
# two centrelines -- u(y) on the vertical one, v(x) on the horizontal one.
# Particles within half a smoothing length of the line are binned, which is the
# nearest thing to "the profile" a particle method has.
positions = runningState.state.positions.detach().cpu().numpy()
velocities = runningState.state.velocities.detach().cpu().numpy()
fluid = (runningState.state.kinds == 0).detach().cpu().numpy()
band = float(ctx.config.dx) * 2

figure, axis = plt.subplots(1, 2, figsize=(11, 4))
near = fluid & (np.abs(positions[:, 0]) < band)
order = np.argsort(positions[near, 1])
axis[0].plot(velocities[near][order, 0], positions[near][order, 1], lw=0.8)
axis[0].set_xlabel('u'); axis[0].set_ylabel('y'); axis[0].set_title('vertical centreline')

near = fluid & (np.abs(positions[:, 1]) < band)
order = np.argsort(positions[near, 0])
axis[1].plot(positions[near][order, 0], velocities[near][order, 1], lw=0.8)
axis[1].set_xlabel('x'); axis[1].set_ylabel('v'); axis[1].set_title('horizontal centreline')
for panel in axis:
    panel.axhline(0, color='black', lw=0.5, ls=':')
    panel.axvline(0, color='black', lw=0.5, ls=':')
figure.tight_layout()

## The usual weakly compressible check

In [ ]:
# The two numbers worth reading off any weakly compressible run: the density
# has to stay within about a percent of `rho0` (that is the whole premise of
# the scheme), and the kinetic energy says whether the flow is doing what it
# was set up to do. Both come from `weaklyCompressibleDiagnostics`, recorded
# every step in the loop above.
figure, axis = plt.subplots(1, 2, figsize=(11, 3.5))
t = [row['t'] for row in trajectory]
axis[0].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[0].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[0].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[0].set_xlabel('t'); axis[0].set_ylabel(r'$\rho$'); axis[0].legend()
axis[1].plot(t, [row['kineticEnergy'] for row in trajectory])
axis[1].set_xlabel('t'); axis[1].set_ylabel('kinetic energy')
figure.tight_layout()